In [1]:
import pandas as pd
import requests
import json
import time

In [ ]:
def fetch_historical_weather_data(latitude, longitude, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "timezone": "Europe/Brussels",
        "wind_speed_unit": "ms",
        "hourly": [
            "temperature_2m",
            "apparent_temperature",
            "relative_humidity_2m",
            "precipitation",
            "rain",
            "snowfall",
            "wind_speed_10m", 
            "shortwave_radiation",
            "direct_normal_irradiance",
            "sunshine_duration"
        ],
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching data: {response.status_code}, {response.reason}")
        return None

In [ ]:
data = fetch_historical_weather_data(50.91618, 4.45612, "2024-01-01", "2024-01-01")
data

In [ ]:
def parse_weather_data_response(site_id, site_name, raw):
    # Metadata of the weather station from the weather API response
    meta = {
        "site_id": site_id,
        "site_name": site_name,
        "latitude": raw["latitude"],
        "longitude": raw["longitude"],
        "elevation_m": raw["elevation"],
        "timezone": raw["timezone"],
    }

    # Hourly weather data as a DataFrame
    df = pd.DataFrame(raw["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    df.insert(0, "site_id", site_id)
    df = df.set_index(["time"])

    return meta, df

In [ ]:
meta, df = parse_weather_data_response(1, "Test Site", data)
meta

In [ ]:
df.head()

## Open-Meteo API

The new fetch function is obtained from the documentation of the Open-Meteo API page by filtering multiple location at once.

The new are not installed in the requirements.txt file, since the data is already downloded and stored in the data folder. However, if you want to run the code, you can install the open-meteo package using commands below:

```
pip install openmeteo-requests
pip install requests-cache retry-requests
```

In [2]:
# pip install openmeteo-requests
# pip install requests-cache retry-requests numpy pandas

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

VARIABLES_OPEN_METEO = [
            "temperature_2m",
            "apparent_temperature",
            "relative_humidity_2m",
            "precipitation",
            "rain",
            "snowfall",
            "wind_speed_10m", 
            "shortwave_radiation",
            "direct_normal_irradiance",
            "sunshine_duration"
        ]

def fetch_historical_weather_data_openmeteo(
		latitudes, 
		longitudes,
		start_date, 
		end_date,
		variables = VARIABLES_OPEN_METEO
	):
	# Setup the Open-Meteo API client with cache and retry on error
	cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
	retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
	openmeteo = openmeteo_requests.Client(session = retry_session)

	# Make sure all required weather variables are listed here
	# The order of variables in hourly or daily is important to assign them correctly below
	url = "https://archive-api.open-meteo.com/v1/archive"
	params = {
		"latitude": latitudes,
		"longitude": longitudes,
		"start_date": start_date,
		"end_date": end_date,
		"timezone": "Europe/Brussels",
        "wind_speed_unit": "ms",
        "hourly": variables
	}
	responses = openmeteo.weather_api(url, params = params)
	return responses

def fetch_historical_weather_forecast_data_openmeteo(
		latitudes, 
		longitudes,
		start_date, 
		end_date,
		variables = VARIABLES_OPEN_METEO
	):
	# Setup the Open-Meteo API client with cache and retry on error
	cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
	retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
	openmeteo = openmeteo_requests.Client(session = retry_session)

	# Make sure all required weather variables are listed here
	# The order of variables in hourly or daily is important to assign them correctly below
	url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
	params = {
		"latitude": latitudes,
		"longitude": longitudes,
		"start_date": start_date,
		"end_date": end_date,
		"timezone": "Europe/Brussels",
        "wind_speed_unit": "ms",
        "hourly": variables
	}
	responses = openmeteo.weather_api(url, params = params)
	return responses


In [ ]:
data = fetch_historical_weather_data_openmeteo(
        latitudes = [50.916183, 51.275120],
        longitudes = [4.456122, 4.471690],
        start_date = "2024-01-01",
        end_date = "2024-01-01"
    )
data

In [3]:
data_forecast = fetch_historical_weather_forecast_data_openmeteo(
        latitudes = [50.916183, 51.275120],
        longitudes = [4.456122, 4.471690],
        start_date = "2024-01-01",
        end_date = "2024-01-01"
    )
data_forecast

In [8]:
data_forecast[0].Hourly().Variables(0).ValuesAsNumpy()

array([7.85, 8.25, 7.95, 6.55, 7.05, 7.35, 6.9 , 6.95, 6.75, 6.5 , 6.15,
       6.15, 7.65, 8.25, 8.9 , 8.05, 7.95, 7.65, 7.2 , 7.15, 7.  , 6.85,
       7.  , 7.35], dtype=float32)

In [6]:
def parse_weather_data_response_openmeteo(site_id, site_name, response):
    # Metadata of the weather station from the weather API response
    meta = {
        "site_id": site_id,
        "site_name": site_name,
        "latitude": response.Latitude(),
        "longitude": response.Longitude(),
        "elevation_m": response.Elevation(),
        "timezone": response.Timezone(),
        "utc_offset_seconds": response.UtcOffsetSeconds(),
    }

    # Hourly weather data as a DataFrame
    hourly = response.Hourly()
    hourly_data = {}
    for i, var in enumerate(VARIABLES_OPEN_METEO):
        hourly_data[var] = hourly.Variables(i).ValuesAsNumpy()

    df = pd.DataFrame(hourly_data)
    
    df["time"] = pd.date_range(
        start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s"),
        end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s"),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    )

    df.insert(0, "site_id", site_id)
    df = df.set_index("time")

    return meta, df

In [7]:
parse_weather_data_response_openmeteo(1, "Test Site", data_forecast[0])

({'site_id': 1,
  'site_name': 'Test Site',
  'latitude': 50.90800094604492,
  'longitude': 4.466000080108643,
  'elevation_m': 24.0,
  'timezone': b'Europe/Brussels',
  'utc_offset_seconds': 7200},
                      site_id  temperature_2m  apparent_temperature  \
 time                                                                 
 2024-01-01 00:00:00        1            7.85              0.968274   
 2024-01-01 01:00:00        1            8.25              1.397761   
 2024-01-01 02:00:00        1            7.95              1.164671   
 2024-01-01 03:00:00        1            6.55              0.291983   
 2024-01-01 04:00:00        1            7.05              0.588700   
 2024-01-01 05:00:00        1            7.35              0.941221   
 2024-01-01 06:00:00        1            6.90              0.485304   
 2024-01-01 07:00:00        1            6.95              0.604391   
 2024-01-01 08:00:00        1            6.75              0.460742   
 2024-01-01 09:00:00

## Get Station IDs and Locations

Column names were provided by the fietstellingen dataset.
### `sites.csv`
 
|  Kolom    | English      | type       | beschrijving              |
|---        |---           |---         |---                        |
| site ID   | site_id      | int        | ID (primary key)          |
| site nr   | site_nr      | int        | Ecocounter ID van site    |
| long      | longitude    | float      | WGS84 longitude           |
| lat       | latitude     | float      | WGS84 Latitude            |
| naam      | name         | text       | naam site                 |
| domein    | domain       | text       | domein naam van site      |
| wegnr     | road_nr      | text       | Wegnummer van site        |
| district  | district     | text       | Nummer wegendistrict      |
| gemeente  | municipality | text       | Gemeente van site         |
| interval  | interval     | int        | meetinterval in minuten   |
| datum_van | date_from    | datum      | datum van installatie     |

### `richtingen.csv`
|  Kolom    | English      | type       | beschrijving              |
|---        |---           |---         |---                        |
| site ID   | site_id      | int        | ID (primary key)          |
| richting  | direction    | Tekst      | IN, OUT of IN/OUT         |
| naam      | name         | text       | Omschrijving richting     |


### `data-<jaar>-<maand>.csv`

|  Kolom    | English      | type       | beschrijving              |
|---        |---           |---         |---                        |
| site ID   | site_id      | int        | FKey naar Site            |
| richting  | direction    | Tekst      | IN, OUT of IN/OUT         |
| type      | type         | Tekst      | Type telling              |
| van       | from         | datum/tijd | start van interval        |
| tot       | to           | datum/tijd | einde van interval        |
| aantal    | count        | int        | aantal geteld in interval |

Type telling kan zijn: VOETGANGERS, FIETSERS, PAARDEN, AUTOS, BUSSEN, 
MINIBUSSEN, NIET GEDEFINIEERD, MOTORFIETSEN, KAYAKS

In [4]:
sites = pd.read_csv('data/sites.csv', header=None)
sites.columns = columns=['site_id', 'site_nr', 'long', 'lat', 'name', 'domain', 'road_nr', 'district', 'municipality', 'interval', 'date_from']
sites.head()

,site_id,site_nr,long,lat,name,domain,road_nr,district,municipality,interval,date_from
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,2019-08-22
2,3,100052863,4.472220,51.275030,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,2019-08-22
3,4,100052864,5.190110,51.160230,Balen 1,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22
4,5,100052865,5.190030,51.160180,Balen 2,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22


In [10]:
sites.site_id.unique()

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 141,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 142, 143, 144,
       145, 146, 147, 148, 149, 150, 151, 152])

In [11]:
sites.iloc[0:2]

,site_id,site_nr,long,lat,name,domain,road_nr,district,municipality,interval,date_from
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,2019-08-22


In [ ]:
# weather_metadata_list = []
# weather_dataframes_list = []

# for index, row in sites.iterrows():
#     latitude = row['lat']
#     longitude = row['long']
#     site_id = row['site_id']
#     site_name = row['name']
#     print(f"Fetching data for site {site_name} at ({round(latitude, 5)}, {round(longitude, 5)})")
#     data = fetch_historical_weather_data(latitude, longitude, "2024-01-01", "2026-03-31")

#     if data:
#         parsed_meta, parsed_df = parse_weather_data_response(site_id, site_name, data)
#         weather_metadata_list.append(parsed_meta)
#         weather_dataframes_list.append(parsed_df)
#     else:
#         print(f"Failed to fetch data for site {site_name}")
#         continue
#     time.sleep(0.5)  # Sleep for 0.5 seconds to avoid hitting API rate limits
# weather_metadata_df = pd.DataFrame(weather_metadata_list)
# weather_df = pd.concat(weather_dataframes_list)

# print("Weather metadata:")
# print(weather_metadata_df.info())

In [ ]:
# weather_df.site_id.unique()

In [14]:
weather_metadata_list = []
weather_dataframes_list = []

CHUNK_SIZE = 10
chunks = [sites.iloc[i:i+CHUNK_SIZE] for i in range(0, len(sites), CHUNK_SIZE)]

for chunk_idx in range(len(chunks)):
    chunk = chunks[chunk_idx]

    print(f"\nChunk {chunk_idx + 1}/{len(chunks)} — {len(chunk)} sites...")

    responses = fetch_historical_weather_data_openmeteo(
        latitudes=chunk["lat"].tolist(),
        longitudes=chunk["long"].tolist(),
        start_date="2023-01-01",
        end_date="2023-12-31",
    )

    for i, response in enumerate(responses):
        site_id   = chunk.iloc[i]["site_id"]
        site_name = chunk.iloc[i]["name"]

        print(f"Parsing response for {site_name}")
        meta, df = parse_weather_data_response_openmeteo(site_id, site_name, response)
        weather_metadata_list.append(meta)
        weather_dataframes_list.append(df)

    # 'Minutely API request limit exceeded. Please try again in one minute.'
    # In order to avoid hitting the API rate limits, sleep for 65 seconds after each chunk of 10 sites
    time.sleep(65)  
weather_metadata_df = pd.DataFrame(weather_metadata_list)
weather_df = pd.concat(weather_dataframes_list)

print("Weather metadata:")
print(weather_metadata_df.info())


Chunk 1/16 — 10 sites...
Parsing response for Machelen
Parsing response for Brasschaat 2
Parsing response for Brasschaat 1
Parsing response for Balen 1
Parsing response for Balen 2
Parsing response for Evergem 2
Parsing response for Evergem 1
Parsing response for Heist op den Berg 1
Parsing response for Heist op den Berg 2
Parsing response for Aalst 2

Chunk 2/16 — 10 sites...
Parsing response for Aalst 3
Parsing response for Tervuren
Parsing response for Gent
Parsing response for Genk
Parsing response for Oostende 2
Parsing response for Kortrijk 1
Parsing response for Kortrijk 2
Parsing response for Oostende 1
Parsing response for Aalst 1
Parsing response for Brugge

Chunk 3/16 — 10 sites...
Parsing response for Bilzen
Parsing response for Tienen 2
Parsing response for Tienen 3
Parsing response for Tienen 1
Parsing response for Bree
Parsing response for Zaventem
Parsing response for Grimbergen
Parsing response for Heers
Parsing response for Tongeren
Parsing response for Herk De Stad


In [16]:
weather_metadata_df

,site_id,site_name,latitude,longitude,elevation_m,timezone,utc_offset_seconds
0,1,Machelen,50.931458,4.500000,24.0,b'Europe/Brussels',7200
1,2,Brasschaat 2,51.282951,4.540540,12.0,b'Europe/Brussels',7200
2,3,Brasschaat 1,51.282951,4.540540,12.0,b'Europe/Brussels',7200
3,4,Balen 1,51.142353,5.170557,32.0,b'Europe/Brussels',7200
4,5,Balen 2,51.142353,5.170557,32.0,b'Europe/Brussels',7200
...,...,...,...,...,...,...,...
146,148,Hoeilaart,50.790859,4.483986,97.0,b'Europe/Brussels',7200
147,149,Machelen,50.861156,4.491979,52.0,b'Europe/Brussels',7200
148,150,Bertem,50.861156,4.491979,72.0,b'Europe/Brussels',7200
149,151,Leuven,50.861156,4.652407,35.0,b'Europe/Brussels',7200


In [9]:
sites.site_id.unique()

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 141,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 142, 143, 144,
       145, 146, 147, 148, 149, 150, 151, 152])

In [10]:
weather_df.site_id.unique()

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 141,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 142, 143, 144,
       145, 146, 147, 148, 149, 150, 151, 152])

In [15]:
weather_df.columns

Index(['site_id', 'temperature_2m', 'apparent_temperature',
       'relative_humidity_2m', 'precipitation', 'rain', 'snowfall',
       'wind_speed_10m', 'shortwave_radiation', 'direct_normal_irradiance',
       'sunshine_duration'],
      dtype='str')

In [17]:
weather_df.to_csv("data/weather_data_2023.csv")

In [18]:
weather_metadata_df.to_csv("data/weather_metadata_2023.csv")

## Merge All Years to Parquet

In [1]:
import pandas as pd

w_2023 = pd.read_csv("data/weather_forecast_data_2023.csv", index_col=0, parse_dates=["time"])
w_2024_2026 = pd.read_csv("data/weather_forecast_data_2024-2026.csv", index_col=0, parse_dates=["time"])

w_all = pd.concat([w_2023, w_2024_2026])

w_all.to_parquet("data/weather_forecast_data_2023-2026.parquet")

In [2]:
w_all.head()

,site_id,temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,shortwave_radiation,direct_normal_irradiance,sunshine_duration
time,,,,,,,,,,,
2023-01-01 00:00:00,1,15.45,9.862126,52.046814,0.0,0.0,0.0,8.591274,0.0,0.0,0.0
2023-01-01 01:00:00,1,15.20,9.478119,51.262646,0.0,0.0,0.0,8.660831,0.0,0.0,0.0
2023-01-01 02:00:00,1,14.95,8.725530,48.922974,0.0,0.0,0.0,9.261749,0.0,0.0,0.0
2023-01-01 03:00:00,1,14.35,8.613325,53.587060,0.1,0.1,0.0,8.628441,0.0,0.0,0.0
2023-01-01 04:00:00,1,14.45,8.578276,54.932552,0.0,0.0,0.0,9.069730,0.0,0.0,0.0


## Get Weather Data from Hugging Face Storage

In [2]:
from huggingface_hub import hf_hub_download
import pandas as pd

# weather_data_df = pd.read_csv(hf_hub_download(
#     repo_id="MDA-Project-Group3/MDA_Project_Data",
#     filename="weather_data_2024-2026.csv",
#     repo_type="dataset",
# ))
# weather_metadata_df = pd.read_csv(hf_hub_download(
#     repo_id="MDA-Project-Group3/MDA_Project_Data",
#     filename="weather_metadata_2024-2026.csv",
#     repo_type="dataset",
# ), index_col=0)

In [4]:
weather_data_df = pd.read_csv("data/weather_data_2024-2026.csv", index_col=0)
weather_data_df.sort_index()

,site_id,temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,shortwave_radiation,direct_normal_irradiance,sunshine_duration
time,,,,,,,,,,,
2024-01-01 00:00:00,1,7.85,0.968274,72.439610,0.0,0.0,0.0,10.104455,0.0,0.0,0.0
2024-01-01 00:00:00,17,7.65,1.814975,77.416374,0.0,0.0,0.0,8.386895,0.0,0.0,0.0
2024-01-01 00:00:00,131,8.05,1.570176,74.810100,0.0,0.0,0.0,9.577578,0.0,0.0,0.0
2024-01-01 00:00:00,22,7.50,0.742469,74.450640,0.0,0.0,0.0,9.885848,0.0,0.0,0.0
2024-01-01 00:00:00,136,8.30,1.800590,73.036070,0.0,0.0,0.0,9.577578,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2026-03-31 23:00:00,36,7.00,4.532538,81.217280,0.0,0.0,0.0,1.991231,0.0,0.0,0.0
2026-03-31 23:00:00,114,7.00,5.540520,91.119560,0.0,0.0,0.0,0.728011,0.0,0.0,0.0
2026-03-31 23:00:00,37,6.15,3.386857,84.012320,0.0,0.0,0.0,2.423324,0.0,0.0,0.0


In [ ]:
wea